# Module 4 — SeqLigase BiLSTM: SMILES Sequence Infilling

**Goal:** reconstruct a contiguous masked fragment in a SMILES string with a standalone PyTorch BiLSTM.

**Dataset:** [Clean QM9 Dataset for SeqLigase](https://www.kaggle.com/datasets/ssshaun/clean-qm9-dataset-fro-seqligase-project) — ~107k RDKit-validated, de-duplicated QM9 molecules with an 80/10/10 leakage-checked split (`seed=42`).

**Kaggle setup**
1. Open this notebook on Kaggle.
2. Right sidebar → **Accelerator** → `GPU T4 x2` (a single T4 is enough).
3. **Internet** → On.
4. **Add Data** → search for `Clean QM9 Dataset for SeqLigase Project` → Add. The notebook also tries `kagglehub` automatically.

**What this notebook produces**
- Modular source files written into the working directory: `config.py`, `data_utils.py`, `models/bilstm.py`, `train.py`, `predict.py`, `export_model.py`
- `vocab.json` with `<PAD>`, `<UNK>`, `<MASK>`, `<SOS>`, `<EOS>`
- `checkpoints/seqligase_bilstm_best.pt` and `checkpoints/seqligase_bilstm_last.pt`
- `artifacts/plots/loss_curve.png` and `artifacts/plots/accuracy_curve.png`
- `artifacts/history.json`, `artifacts/test_metrics.json`
- `artifacts/exports/seqligase_bilstm_export.pt`, `.zip`, `vocab.json`, `MODEL_CARD.md`
- Interactive CLI (`predict.py`) and the OpenRouter-ready `get_bilstm_predictions()` wrapper

**Expected runtime:** roughly 25–60 minutes for 30 epochs on a T4 with the full dataset (early stopping often finishes sooner). For a quick smoke test, set `epochs=2, max_samples=5000` in the training cell.

Run the cells top-to-bottom. All generated files land in `/kaggle/working`, where Kaggle exposes them as notebook output.


## 1. Environment setup

Kaggle already provides `torch`, `matplotlib`, `pandas`, and `scikit-learn`. This cell installs `rdkit` / `kagglehub` only when they are missing.


In [ ]:
import importlib.util
import platform
import subprocess
import sys
from pathlib import Path

for package, module in (("kagglehub", "kagglehub"), ("rdkit", "rdkit")):
    if importlib.util.find_spec(module) is None:
        print(f"Installing {package} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=False,
        )

import torch

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("Project root:", PROJECT_ROOT)


## 2. Scaffold the modular project

The notebook writes the same modules you would keep in a normal repository. Later cells import them, so the logic stays modular while the notebook remains a single portable file.


In [ ]:
from pathlib import Path

for directory in (
    "models",
    "data/raw",
    "checkpoints",
    "artifacts/plots",
    "artifacts/exports",
):
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Project scaffold ready at:", Path.cwd())


In [ ]:
%%writefile config.py
"""Central configuration for the SeqLigase BiLSTM SMILES-infilling project."""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PLOT_DIR = ARTIFACT_DIR / "plots"
EXPORT_DIR = ARTIFACT_DIR / "exports"
VOCAB_PATH = PROJECT_ROOT / "vocab.json"

BEST_CHECKPOINT_NAME = "seqligase_bilstm_best.pt"
LAST_CHECKPOINT_NAME = "seqligase_bilstm_last.pt"

# ---------------------------------------------------------------------------
# Vocabulary / tokenizer
# ---------------------------------------------------------------------------
SPECIAL_TOKENS: tuple[str, ...] = (
    "<PAD>",
    "<UNK>",
    "<MASK>",
    "<SOS>",
    "<EOS>",
)
PAD_TOKEN, UNK_TOKEN, MASK_TOKEN, SOS_TOKEN, EOS_TOKEN = SPECIAL_TOKENS
PAD_ID, UNK_ID, MASK_ID, SOS_ID, EOS_ID = range(len(SPECIAL_TOKENS))

# The model should never emit structural tokens as a reconstruction. These
# ids are excluded from the softmax in both training and inference.
OUTPUT_BANNED_IDS: tuple[int, ...] = (PAD_ID, UNK_ID, MASK_ID, SOS_ID, EOS_ID)

IGNORE_INDEX = -100

# Placeholders accepted by the inference CLI. ``*`` is the common wildcard
# notation used in SMILES examples and is treated as a mask.
MASK_PLACEHOLDERS: tuple[str, ...] = (
    "[MASK]",
    "<MASK>",
    "[mask]",
    "<mask>",
    "*",
    "[*]",
)

# ---------------------------------------------------------------------------
# Kaggle dataset
# ---------------------------------------------------------------------------
KAGGLE_DATASET_SLUG = "ssshaun/clean-qm9-dataset-fro-seqligase-project"
SMILES_COLUMN_CANDIDATES: tuple[str, ...] = (
    "canonical_smiles",
    "smiles",
    "original_smiles",
    "smiles_string",
    "smi",
    "mol",
)


def ensure_project_dirs() -> None:
    """Create every directory the pipeline writes to."""
    for directory in (
        DATA_DIR,
        RAW_DATA_DIR,
        CHECKPOINT_DIR,
        ARTIFACT_DIR,
        PLOT_DIR,
        EXPORT_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------------
@dataclass
class TrainConfig:
    """Hyper-parameters shared by training, evaluation, and inference."""

    seed: int = 42

    # Model
    embed_dim: int = 192
    hidden_dim: int = 384
    num_layers: int = 3
    dropout: float = 0.2

    # Sequence / masking
    max_len: int = 128
    max_mask_span: int = 6
    # >1 biases mask spans toward short fragments: span ~ 1 + int(max * u**p).
    mask_span_power: float = 4.0

    # Optimization
    batch_size: int = 256
    epochs: int = 30
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    scheduler_factor: float = 0.5
    scheduler_patience: int = 3
    early_stopping_patience: int = 8

    # Runtime
    num_workers: int = 2
    amp: bool = True
    rebuild_vocab: bool = False
    max_samples: int = 0  # 0 == use the full dataset

    # Output
    checkpoint_name: str = BEST_CHECKPOINT_NAME

    def to_dict(self) -> dict:
        """Return a JSON-serialisable dictionary."""
        return asdict(self)


In [ ]:
%%writefile data_utils.py
"""Data download, tokenisation, vocabulary, masking, and splitting utilities."""

from __future__ import annotations

import json
import random
import re
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import pandas as pd
import torch
from rdkit import Chem, RDLogger
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from config import (
    IGNORE_INDEX,
    KAGGLE_DATASET_SLUG,
    MASK_ID,
    PAD_ID,
    RAW_DATA_DIR,
    SMILES_COLUMN_CANDIDATES,
    SOS_ID,
    EOS_ID,
    SPECIAL_TOKENS,
    UNK_ID,
    ensure_project_dirs,
)

RDLogger.DisableLog("rdApp.*")

# Keep bracketed atoms (e.g. ``[C@@H]``), two-letter elements, chirality
# markers and ring-closure percentages as single tokens.
TOKEN_PATTERN = re.compile(
    r"\[[^\]]+\]|Br|Cl|Si|Se|Na|Li|Mg|Ca|Al|@@|@|%\d{2}|.",
    re.DOTALL,
)


# ---------------------------------------------------------------------------
# Tokenisation / RDKit helpers
# ---------------------------------------------------------------------------
def tokenize_smiles(smiles: str) -> list[str]:
    """Split a SMILES string into chemical tokens."""
    if not isinstance(smiles, str):
        return []
    return TOKEN_PATTERN.findall(smiles.strip())


def detokenize_smiles(tokens: Iterable[str]) -> str:
    """Join tokens back into a SMILES string."""
    return "".join(tokens)


def is_valid_smiles(smiles: str) -> bool:
    """Return ``True`` when RDKit can parse the SMILES string."""
    if not isinstance(smiles, str) or not smiles.strip():
        return False
    return Chem.MolFromSmiles(smiles) is not None


def canonicalize_smiles(smiles: str) -> str | None:
    """Return the RDKit canonical SMILES, or ``None`` when invalid."""
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        return None
    return Chem.MolToSmiles(molecule)


# ---------------------------------------------------------------------------
# Vocabulary
# ---------------------------------------------------------------------------
def build_vocab(smiles: Iterable[str], min_freq: int = 1) -> dict[str, Any]:
    """Build a deterministic token vocabulary from a collection of SMILES."""
    counter: Counter[str] = Counter()
    for value in smiles:
        counter.update(tokenize_smiles(value))

    trained_tokens = [
        token
        for token, frequency in counter.items()
        if frequency >= min_freq and token not in SPECIAL_TOKENS
    ]
    trained_tokens.sort(key=lambda token: (-counter[token], token))

    token_to_id: dict[str, int] = {
        token: index for index, token in enumerate(SPECIAL_TOKENS)
    }
    for token in trained_tokens:
        token_to_id[token] = len(token_to_id)

    id_to_token = [token for token, _ in sorted(token_to_id.items(), key=lambda kv: kv[1])]

    return {
        "special_tokens": list(SPECIAL_TOKENS),
        "token_to_id": token_to_id,
        "id_to_token": id_to_token,
        "tokenizer_pattern": TOKEN_PATTERN.pattern,
        "min_freq": int(min_freq),
        "num_tokens": len(id_to_token),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }


def save_vocab(vocab: dict[str, Any], path: str | Path) -> Path:
    """Persist vocabulary mappings to JSON."""
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(vocab, indent=2, ensure_ascii=False), encoding="utf-8")
    return output


def load_vocab(path: str | Path) -> dict[str, Any]:
    """Load a vocabulary saved by :func:`save_vocab`."""
    source = Path(path)
    if not source.exists():
        raise FileNotFoundError(f"Vocabulary not found: {source}")
    vocab = json.loads(source.read_text(encoding="utf-8"))
    id_to_token = vocab.get("id_to_token", [])
    if isinstance(id_to_token, dict):  # tolerate older JSON with string keys
        id_to_token = [id_to_token[str(i)] for i in range(len(id_to_token))]
    vocab["id_to_token"] = id_to_token
    return vocab


def load_or_build_vocab(
    train_smiles: Sequence[str],
    path: str | Path,
    min_freq: int = 1,
    force: bool = False,
) -> dict[str, Any]:
    """Load ``vocab.json`` when present, otherwise build and save it."""
    target = Path(path)
    if target.exists() and not force:
        return load_vocab(target)
    vocab = build_vocab(train_smiles, min_freq=min_freq)
    save_vocab(vocab, target)
    return vocab


# ---------------------------------------------------------------------------
# Dataset loading / column detection / splitting
# ---------------------------------------------------------------------------
def discover_csv_files(data_dir: str | Path) -> list[Path]:
    """Recursively find CSV files under ``data_dir``."""
    root = Path(data_dir)
    if not root.exists():
        return []
    return sorted(root.rglob("*.csv"))


def identify_smiles_column(frame: pd.DataFrame) -> str:
    """Find the SMILES column using names first and RDKit validity as fallback."""
    lower_to_original = {str(column).strip().lower(): column for column in frame.columns}
    for candidate in SMILES_COLUMN_CANDIDATES:
        if candidate in lower_to_original:
            return lower_to_original[candidate]

    for lowered, original in lower_to_original.items():
        if "smiles" in lowered:
            return original

    best_column: str | None = None
    best_score = -1.0
    for column in frame.select_dtypes(include=["object"]).columns:
        sample = frame[column].dropna().astype(str).head(200)
        if sample.empty:
            continue
        score = sum(is_valid_smiles(value) for value in sample) / len(sample)
        if score > best_score:
            best_column, best_score = column, score

    if best_column is None or best_score <= 0.0:
        raise ValueError(
            "Could not identify a SMILES column. Expected one of "
            f"{SMILES_COLUMN_CANDIDATES}; found {list(frame.columns)}."
        )
    return best_column


def load_smiles_dataframe(data_dir: str | Path = RAW_DATA_DIR) -> pd.DataFrame:
    """Load and clean the dataset, returning at least a ``smiles`` column.

    When ``qm9_clean.csv`` is present it is preferred because it contains the
    full table (including the leakage-checked ``split`` column). Otherwise all
    discoverable CSVs are concatenated.
    """
    root = Path(data_dir)
    csv_files = discover_csv_files(root)
    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found under {root}. Run download_kaggle_dataset() "
            "or attach the Kaggle dataset to the notebook."
        )

    preferred = [path for path in csv_files if path.name.lower() == "qm9_clean.csv"]
    selected = preferred if preferred else csv_files

    frames: list[pd.DataFrame] = []
    for path in selected:
        try:
            frame = pd.read_csv(path)
        except Exception as exc:  # pragma: no cover - defensive
            print(f"[warn] Could not read {path}: {exc}")
            continue
        if frame.empty:
            continue
        try:
            smiles_column = identify_smiles_column(frame)
        except ValueError as exc:
            print(f"[warn] Skipping {path.name}: {exc}")
            continue

        frame = frame.copy()
        if smiles_column != "smiles":
            if "smiles" in frame.columns:
                frame = frame.drop(columns=["smiles"])
            frame = frame.rename(columns={smiles_column: "smiles"})

        keep = ["smiles"]
        if "split" in frame.columns:
            keep.append("split")
        frames.append(frame[keep])

    if not frames:
        raise ValueError("No usable SMILES table was found in the dataset.")

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.dropna(subset=["smiles"])
    combined["smiles"] = combined["smiles"].astype(str).str.strip()
    combined = combined[combined["smiles"] != ""]
    combined = combined.drop_duplicates(subset=["smiles"], keep="first")
    return combined.reset_index(drop=True)


def filter_smiles_dataframe(frame: pd.DataFrame, max_len: int = 128) -> pd.DataFrame:
    """Drop SMILES that exceed the model's token budget."""
    lengths = frame["smiles"].map(lambda value: len(tokenize_smiles(value)))
    kept = frame.loc[(lengths >= 1) & (lengths <= max_len - 2)].copy()
    kept["token_length"] = lengths.loc[kept.index]
    return kept.reset_index(drop=True)


def split_dataframe(
    frame: pd.DataFrame,
    seed: int = 42,
    use_provided_split: bool = True,
) -> dict[str, pd.DataFrame]:
    """Create a reproducible 80/10/10 train/validation/test split.

    The Kaggle dataset already ships a leakage-checked split. When its
    ``split`` column is available it is reused; otherwise scikit-learn
    performs a seeded two-stage split.
    """
    if use_provided_split and "split" in frame.columns:
        labels = frame["split"].astype(str).str.strip().str.lower()
        mapping = {
            "train": "train",
            "training": "train",
            "val": "val",
            "valid": "val",
            "validation": "val",
            "test": "test",
            "testing": "test",
        }
        mapped = labels.map(mapping)
        if mapped.notna().all() and {"train", "val", "test"}.issubset(
            set(mapped.unique())
        ):
            parts = {
                name: frame.loc[mapped == name].reset_index(drop=True)
                for name in ("train", "val", "test")
            }
            if all(not part.empty for part in parts.values()):
                return parts

    train_frame, remainder = train_test_split(
        frame, test_size=0.20, random_state=seed, shuffle=True
    )
    val_frame, test_frame = train_test_split(
        remainder, test_size=0.50, random_state=seed, shuffle=True
    )
    return {
        "train": train_frame.reset_index(drop=True),
        "val": val_frame.reset_index(drop=True),
        "test": test_frame.reset_index(drop=True),
    }
# ---------------------------------------------------------------------------
# Dynamic masking dataset
# ---------------------------------------------------------------------------
class MaskedSMILESDataset(Dataset):
    """Generate ``(masked_input, reconstruction_target)`` pairs on the fly.

    A contiguous span of tokens is replaced with ``<MASK>`` tokens in the
    input. The target keeps the original token ids at those positions and
    ``-100`` (ignored by ``CrossEntropyLoss``) everywhere else. Training is
    therefore focused on the fragment that must be inferred.

    When ``dynamic`` is ``True``, :meth:`set_epoch` changes the mask location
    every epoch. Validation and test datasets should set ``dynamic=False`` so
    that reported metrics are stable across epochs.

    ``mask_span_power`` biases the span distribution toward short fragments:
    ``span = 1 + int(max_span * u**power)`` for ``u ~ U(0, 1)``. A power of
    1.0 is uniform; with ``max_span=6`` and ``power=4.0`` roughly 64% of masks
    cover a single token and ~12% cover two, which noticeably lifts token
    accuracy without making the task trivial.

    Token ids are pre-computed once in :meth:`__init__` and reused for every
    epoch, so masking no longer re-tokenises the corpus each time.
    """

    def __init__(
        self,
        smiles: Sequence[str],
        vocab: dict[str, Any],
        max_len: int = 128,
        max_mask_span: int = 6,
        mask_span_power: float = 4.0,
        seed: int = 42,
        dynamic: bool = True,
    ) -> None:
        self.smiles = [
            str(value).strip()
            for value in smiles
            if isinstance(value, str) and str(value).strip()
        ]
        if not self.smiles:
            raise ValueError("MaskedSMILESDataset received no valid SMILES.")
        self.token_to_id: dict[str, int] = vocab["token_to_id"]
        self.max_len = int(max_len)
        self.max_mask_span = int(max_mask_span)
        self.mask_span_power = max(float(mask_span_power), 1e-6)
        self.seed = int(seed)
        self.dynamic = bool(dynamic)
        self.epoch = 0

        # Cache the [SOS] ... [EOS] token ids once; masking only copies them.
        self._cached_ids: list[list[int]] = []
        for value in self.smiles:
            tokens = tokenize_smiles(value)[: max(1, self.max_len - 2)]
            token_ids = [SOS_ID]
            token_ids.extend(self.token_to_id.get(token, UNK_ID) for token in tokens)
            token_ids.append(EOS_ID)
            self._cached_ids.append(token_ids)

    def __len__(self) -> int:
        return len(self.smiles)

    def set_epoch(self, epoch: int) -> None:
        """Change the masking RNG stream (used to refresh training masks)."""
        self.epoch = int(epoch) if self.dynamic else 0

    def _rng(self, index: int) -> random.Random:
        stream = (
            self.seed * 1_000_003
            + index * 9_176
            + self.epoch * 104_729
        ) % (2**32)
        return random.Random(stream)

    def __getitem__(self, index: int) -> dict[str, Any]:
        raw_smiles = self.smiles[index]
        token_ids = list(self._cached_ids[index])

        n_interior = len(token_ids) - 2
        if n_interior < 1:  # defensive: never expected after filtering
            token_ids = [SOS_ID, UNK_ID, EOS_ID]
            n_interior = 1

        rng = self._rng(index)
        max_span = min(self.max_mask_span, max(1, n_interior - 1))
        span = 1 + int(max_span * (rng.random() ** self.mask_span_power))
        span = min(max(span, 1), max_span)

        start = rng.randint(1, n_interior - span + 1)
        end = start + span

        input_ids = list(token_ids)
        labels = [IGNORE_INDEX] * len(token_ids)
        for position in range(start, end):
            input_ids[position] = MASK_ID
            labels[position] = token_ids[position]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "attention_mask": torch.ones(len(token_ids), dtype=torch.long),
            "original_ids": torch.tensor(token_ids, dtype=torch.long),
            "mask_start": start,
            "mask_end": end,
            "mask_length": span,
            "original_smiles": raw_smiles,
        }


def collate_masked_batch(
    batch: list[dict[str, Any]],
    pad_id: int = PAD_ID,
) -> dict[str, Any]:
    """Right-pad variable-length examples for the BiLSTM."""
    batch_size = len(batch)
    max_length = max(int(item["input_ids"].numel()) for item in batch)

    input_ids = torch.full((batch_size, max_length), pad_id, dtype=torch.long)
    labels = torch.full((batch_size, max_length), IGNORE_INDEX, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_length), dtype=torch.long)
    original_ids = torch.full((batch_size, max_length), pad_id, dtype=torch.long)

    for row, item in enumerate(batch):
        length = int(item["input_ids"].numel())
        input_ids[row, :length] = item["input_ids"]
        labels[row, :length] = item["labels"]
        attention_mask[row, :length] = item["attention_mask"]
        original_ids[row, :length] = item["original_ids"]

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        "original_ids": original_ids,
        "mask_start": torch.tensor(
            [int(item["mask_start"]) for item in batch], dtype=torch.long
        ),
        "mask_end": torch.tensor(
            [int(item["mask_end"]) for item in batch], dtype=torch.long
        ),
        "mask_length": torch.tensor(
            [int(item["mask_length"]) for item in batch], dtype=torch.long
        ),
        "original_smiles": [item["original_smiles"] for item in batch],
    }


# ---------------------------------------------------------------------------
# Kaggle download helpers
# ---------------------------------------------------------------------------
def _copy_tabular_files(source: Path, destination: Path) -> int:
    """Copy CSV/JSON files (not the large `.npy` fingerprints) recursively."""
    copied = 0
    for path in Path(source).rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {".csv", ".json"}:
            continue
        relative = path.relative_to(source)
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists() or target.stat().st_size != path.stat().st_size:
            shutil.copy2(path, target)
        copied += 1
    return copied


def download_kaggle_dataset(destination: str | Path = RAW_DATA_DIR) -> Path:
    """Resolve the dataset into ``data/raw``.

    Resolution order:

    1. Data already present in ``destination``.
    2. A dataset attached to the Kaggle notebook under ``/kaggle/input``.
    3. ``kagglehub.dataset_download`` (works on Kaggle with credentials).
    4. The ``kaggle`` CLI fallback.
    """
    ensure_project_dirs()
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=True)

    if discover_csv_files(target):
        print(f"[data] Using existing CSV files in {target}")
        return target

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in kaggle_input.iterdir():
            name = candidate.name.lower()
            if "qm9" in name or "seqligase" in name:
                copied = _copy_tabular_files(candidate, target)
                if copied and discover_csv_files(target):
                    print(f"[data] Copied {copied} file(s) from {candidate}")
                    return target

    errors: list[str] = []
    try:
        import kagglehub  # type: ignore

        downloaded = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
        _copy_tabular_files(downloaded, target)
        if discover_csv_files(target):
            print(f"[data] Downloaded dataset with kagglehub to {target}")
            return target
    except Exception as exc:  # pragma: no cover - depends on credentials
        errors.append(f"kagglehub: {exc}")

    try:
        command = [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "download",
            "-d",
            KAGGLE_DATASET_SLUG,
            "-p",
            str(target),
            "--unzip",
        ]
        subprocess.run(command, check=True, capture_output=True, text=True)
        if discover_csv_files(target):
            print(f"[data] Downloaded dataset with the Kaggle CLI to {target}")
            return target
    except Exception as exc:  # pragma: no cover - depends on credentials
        errors.append(f"kaggle CLI: {exc}")

    detail = "\n  ".join(errors) if errors else "no download method available"
    raise RuntimeError(
        "Could not obtain the dataset.\n"
        f"  {detail}\n"
        "Fix: on Kaggle use 'Add Data' and attach "
        f"'{KAGGLE_DATASET_SLUG}', then re-run this cell. Locally run:\n"
        f"  kaggle datasets download -d {KAGGLE_DATASET_SLUG} -p {target} --unzip\n"
        "or set up ~/.kaggle/kaggle.json first."
    )


In [ ]:
%%writefile models/__init__.py
from .bilstm import SeqLigaseBiLSTM

__all__ = ["SeqLigaseBiLSTM"]


In [ ]:
%%writefile models/bilstm.py
"""Bidirectional LSTM model for SMILES sequence infilling."""

from __future__ import annotations

from pathlib import Path
from typing import Any

import torch
from torch import nn
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence


class SeqLigaseBiLSTM(nn.Module):
    """Token-level BiLSTM encoder with a per-position vocabulary head.

    The model reads a (possibly masked) SMILES token sequence and predicts a
    distribution over the vocabulary at every position. During training the
    loss is applied only to masked positions, so the network learns to
    reconstruct the missing fragment from its bidirectional context.

    Parameters
    ----------
    vocab_size:
        Number of tokens, including the five special tokens.
    embed_dim:
        Size of the learned token embeddings.
    hidden_dim:
        Hidden size of each LSTM direction. The output width is
        ``hidden_dim * 2`` because the LSTM is bidirectional.
    num_layers:
        Number of stacked LSTM layers (2 or 3 recommended).
    dropout:
        Dropout applied to embeddings, between LSTM layers, and before the
        output projection.
    pad_idx:
        Padding token id. Its embedding is kept at zero.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ) -> None:
        super().__init__()
        if vocab_size < 2:
            raise ValueError("vocab_size must be at least 2")
        if num_layers < 1:
            raise ValueError("num_layers must be at least 1")

        self.vocab_size = int(vocab_size)
        self.embed_dim = int(embed_dim)
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.pad_idx = int(pad_idx)

        self.embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.embed_dim,
            padding_idx=self.pad_idx,
        )
        self.embed_dropout = nn.Dropout(self.dropout)
        self.lstm = nn.LSTM(
            input_size=self.embed_dim,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=self.dropout if self.num_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(self.hidden_dim * 2)
        self.output_dropout = nn.Dropout(self.dropout)
        self.output_projection = nn.Linear(self.hidden_dim * 2, self.vocab_size)

        self.apply(self._init_weights)
        with torch.no_grad():
            self.embedding.weight[self.pad_idx].zero_()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Return logits of shape ``(batch, seq_len, vocab_size)``."""
        embedded = self.embed_dropout(self.embedding(input_ids))

        if attention_mask is not None:
            lengths = attention_mask.sum(dim=1).clamp(min=1).to("cpu")
            packed = pack_padded_sequence(
                embedded,
                lengths,
                batch_first=True,
                enforce_sorted=False,
            )
            packed_outputs, _ = self.lstm(packed)
            outputs, _ = pad_packed_sequence(
                packed_outputs,
                batch_first=True,
                total_length=input_ids.size(1),
            )
        else:
            outputs, _ = self.lstm(embedded)

        outputs = self.layer_norm(outputs)
        logits = self.output_projection(self.output_dropout(outputs))
        return logits

    @property
    def num_parameters(self) -> int:
        """Number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    @classmethod
    def from_checkpoint(
        cls,
        checkpoint_path: str | Path,
        map_location: str | torch.device = "cpu",
    ) -> tuple["SeqLigaseBiLSTM", dict[str, Any]]:
        """Rebuild a model from a checkpoint produced by :mod:`train`."""
        path = Path(checkpoint_path)
        if not path.exists():
            raise FileNotFoundError(
                f"Checkpoint not found: {path}. Run train.py (or the training "
                "cell of module_4_BiLSTM.ipynb) first."
            )
        try:
            checkpoint = torch.load(
                path, map_location=map_location, weights_only=False
            )
        except TypeError:  # torch < 1.13
            checkpoint = torch.load(path, map_location=map_location)

        model = cls(**checkpoint["model_config"])
        model.load_state_dict(checkpoint["model_state_dict"])
        return model, checkpoint


In [ ]:
%%writefile train.py
"""Training, evaluation, plotting, and checkpointing for SeqLigase."""

from __future__ import annotations

import argparse
import copy
import json
import math
import os
import random
import time
from contextlib import nullcontext
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from rdkit import RDLogger
from sklearn.metrics import precision_score
from torch import nn
from torch.utils.data import DataLoader

from config import (
    ARTIFACT_DIR,
    CHECKPOINT_DIR,
    IGNORE_INDEX,
    LAST_CHECKPOINT_NAME,
    OUTPUT_BANNED_IDS,
    PAD_ID,
    PLOT_DIR,
    RAW_DATA_DIR,
    SPECIAL_TOKENS,
    UNK_TOKEN,
    VOCAB_PATH,
    TrainConfig,
    ensure_project_dirs,
)
from data_utils import (
    MaskedSMILESDataset,
    canonicalize_smiles,
    collate_masked_batch,
    detokenize_smiles,
    download_kaggle_dataset,
    filter_smiles_dataframe,
    load_or_build_vocab,
    load_smiles_dataframe,
    split_dataframe,
)
from models.bilstm import SeqLigaseBiLSTM

RDLogger.DisableLog("rdApp.*")


# ---------------------------------------------------------------------------
# Runtime helpers
# ---------------------------------------------------------------------------
def set_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def resolve_device() -> torch.device:
    """Select CUDA, then Apple MPS, then CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None:
        if torch.backends.mps.is_available():
            return torch.device("mps")
    return torch.device("cpu")


def create_grad_scaler(enabled: bool, device_type: str) -> Any:
    """Create an AMP GradScaler across PyTorch versions."""
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def decode_ids(
    token_ids: list[int] | tuple[int, ...],
    id_to_token: list[str],
    skip_special: bool = True,
) -> str:
    """Decode token ids into a SMILES string, skipping special tokens."""
    tokens: list[str] = []
    for raw_id in token_ids:
        index = int(raw_id)
        token = id_to_token[index] if 0 <= index < len(id_to_token) else UNK_TOKEN
        if skip_special and token in SPECIAL_TOKENS:
            continue
        tokens.append(token)
    return detokenize_smiles(tokens)


def restrict_output_logits(logits: torch.Tensor) -> torch.Tensor:
    """Mask structural special tokens out of the per-position distribution.

    ``<PAD>``, ``<UNK>``, ``<MASK>``, ``<SOS>``, and ``<EOS>`` are never valid
    reconstruction targets, so giving them probability mass only wastes model
    capacity. Setting their logits to a large negative value removes them from
    both the cross-entropy loss and the argmax/top-k metrics.
    """
    if not OUTPUT_BANNED_IDS:
        return logits
    banned = torch.tensor(OUTPUT_BANNED_IDS, device=logits.device, dtype=torch.long)
    return logits.index_fill(-1, banned, -1e4)


def _token_statistics(
    logits: torch.Tensor,
    labels: torch.Tensor,
) -> tuple[int, int, int, np.ndarray, np.ndarray]:
    """Return correct@1, correct@3, token count, gold, and predicted ids."""
    mask = labels.ne(IGNORE_INDEX)
    n_tokens = int(mask.sum())
    if n_tokens == 0:
        empty = np.empty(0, dtype=np.int64)
        return 0, 0, 0, empty, empty

    predictions = logits.argmax(dim=-1)
    gold = labels[mask]
    guessed = predictions[mask]
    correct = int((guessed == gold).sum())

    top3_ids = logits.topk(3, dim=-1).indices
    top3_hits = (top3_ids == labels.unsqueeze(-1)).any(dim=-1) & mask
    top3_correct = int(top3_hits.sum())

    return (
        correct,
        top3_correct,
        n_tokens,
        gold.detach().cpu().numpy(),
        guessed.detach().cpu().numpy(),
    )


def _reconstruction_statistics(
    reconstructed: torch.Tensor,
    original_smiles: list[str],
    id_to_token: list[str],
    check_rdkit: bool,
) -> dict[str, float]:
    """Character similarity plus optional RDKit validity / exact reconstruction."""
    char_scores: list[float] = []
    structural_passes = 0
    valid_predictions = 0
    n_sequences = 0

    for row, reference in zip(reconstructed, original_smiles):
        predicted_smiles = decode_ids(row.tolist(), id_to_token)
        char_scores.append(SequenceMatcher(None, predicted_smiles, reference).ratio())
        n_sequences += 1

        if check_rdkit:
            predicted_canonical = canonicalize_smiles(predicted_smiles)
            reference_canonical = canonicalize_smiles(reference)
            if predicted_canonical is not None:
                valid_predictions += 1
            if (
                predicted_canonical is not None
                and reference_canonical is not None
                and predicted_canonical == reference_canonical
            ):
                structural_passes += 1

    denominator = max(n_sequences, 1)
    return {
        "char_accuracy": float(np.mean(char_scores)) if char_scores else 0.0,
        "structural_pass_rate": structural_passes / denominator,
        "validity_rate": valid_predictions / denominator,
        "structural_passes": float(structural_passes),
        "valid_predictions": float(valid_predictions),
        "n_sequences": float(n_sequences),
    }


def _macro_precision(
    gold_chunks: list[np.ndarray],
    predicted_chunks: list[np.ndarray],
) -> float:
    if not gold_chunks or not predicted_chunks:
        return 0.0
    gold = np.concatenate(gold_chunks)
    predicted = np.concatenate(predicted_chunks)
    if gold.size == 0:
        return 0.0
    return float(precision_score(gold, predicted, average="macro", zero_division=0))


# ---------------------------------------------------------------------------
# Epoch loops
# ---------------------------------------------------------------------------
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: Any,
    device: torch.device,
    config: TrainConfig,
    use_amp: bool,
    id_to_token: list[str],
) -> dict[str, float]:
    """Run one optimisation epoch and return training metrics."""
    model.train()
    total_loss = 0.0
    total_tokens = 0
    correct = 0
    top3_correct = 0
    gold_chunks: list[np.ndarray] = []
    predicted_chunks: list[np.ndarray] = []
    char_scores: list[float] = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        original_ids = batch["original_ids"].to(device, non_blocking=True)

        n_tokens = int(labels.ne(IGNORE_INDEX).sum())
        if n_tokens == 0:
            continue

        optimizer.zero_grad(set_to_none=True)
        autocast_context = (
            torch.autocast(device_type=device.type, enabled=True)
            if use_amp
            else nullcontext()
        )
        with autocast_context:
            logits = model(input_ids, attention_mask)
            logits = restrict_output_logits(logits)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.detach()) * n_tokens
        total_tokens += n_tokens

        with torch.no_grad():
            batch_correct, batch_top3, _, gold, guessed = _token_statistics(
                logits.detach(), labels
            )
            correct += batch_correct
            top3_correct += batch_top3
            gold_chunks.append(gold)
            predicted_chunks.append(guessed)

            mask = labels.ne(IGNORE_INDEX)
            reconstructed = torch.where(mask, logits.argmax(dim=-1), original_ids)
            reconstruction = _reconstruction_statistics(
                reconstructed,
                batch["original_smiles"],
                id_to_token=id_to_token,
                check_rdkit=False,
            )
            char_scores.append(reconstruction["char_accuracy"])

    denominator = max(total_tokens, 1)
    return {
        "loss": total_loss / denominator if total_tokens else 0.0,
        "token_accuracy": correct / denominator if total_tokens else 0.0,
        "top3_accuracy": top3_correct / denominator if total_tokens else 0.0,
        "token_precision": _macro_precision(gold_chunks, predicted_chunks),
        "char_accuracy": float(np.mean(char_scores)) if char_scores else 0.0,
        "n_tokens": float(total_tokens),
    }
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_amp: bool,
    id_to_token: list[str],
    check_rdkit: bool = True,
) -> dict[str, float]:
    """Evaluate a model and return loss plus token and structural metrics."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    correct = 0
    top3_correct = 0
    gold_chunks: list[np.ndarray] = []
    predicted_chunks: list[np.ndarray] = []
    char_scores: list[float] = []
    structural_passes = 0
    valid_predictions = 0
    n_sequences = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        original_ids = batch["original_ids"].to(device, non_blocking=True)

        n_tokens = int(labels.ne(IGNORE_INDEX).sum())
        if n_tokens == 0:
            continue

        autocast_context = (
            torch.autocast(device_type=device.type, enabled=True)
            if use_amp
            else nullcontext()
        )
        with autocast_context:
            logits = model(input_ids, attention_mask)
            logits = restrict_output_logits(logits)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
            )

        total_loss += float(loss) * n_tokens
        total_tokens += n_tokens

        batch_correct, batch_top3, _, gold, guessed = _token_statistics(logits, labels)
        correct += batch_correct
        top3_correct += batch_top3
        gold_chunks.append(gold)
        predicted_chunks.append(guessed)

        mask = labels.ne(IGNORE_INDEX)
        reconstructed = torch.where(mask, logits.argmax(dim=-1), original_ids)
        reconstruction = _reconstruction_statistics(
            reconstructed,
            batch["original_smiles"],
            id_to_token=id_to_token,
            check_rdkit=check_rdkit,
        )
        char_scores.append(reconstruction["char_accuracy"])
        structural_passes += int(reconstruction["structural_passes"])
        valid_predictions += int(reconstruction["valid_predictions"])
        n_sequences += int(reconstruction["n_sequences"])

    denominator = max(total_tokens, 1)
    sequence_denominator = max(n_sequences, 1)
    return {
        "loss": total_loss / denominator if total_tokens else 0.0,
        "token_accuracy": correct / denominator if total_tokens else 0.0,
        "top3_accuracy": top3_correct / denominator if total_tokens else 0.0,
        "token_precision": _macro_precision(gold_chunks, predicted_chunks),
        "char_accuracy": float(np.mean(char_scores)) if char_scores else 0.0,
        "structural_pass_rate": structural_passes / sequence_denominator,
        "validity_rate": valid_predictions / sequence_denominator,
        "n_tokens": float(total_tokens),
        "n_sequences": float(n_sequences),
    }


# ---------------------------------------------------------------------------
# Checkpoints / artefacts
# ---------------------------------------------------------------------------
def analyze_by_span_length(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_amp: bool,
) -> pd.DataFrame:
    """Break test accuracy down by the number of masked tokens per example.

    Short masked spans are easier to reconstruct than long ones, so this table
    makes the trade-off explicit instead of hiding it in one aggregate number.
    """
    model.eval()
    totals: dict[int, dict[str, float]] = {}

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            span_lengths = batch["mask_length"].tolist()

            autocast_context = (
                torch.autocast(device_type=device.type, enabled=True)
                if use_amp
                else nullcontext()
            )
            with autocast_context:
                logits = model(input_ids, attention_mask)
                logits = restrict_output_logits(logits)

            predictions = logits.argmax(dim=-1)
            mask = labels.ne(IGNORE_INDEX)
            matches = (predictions == labels) & mask
            per_example_correct = matches.sum(dim=1).tolist()
            per_example_tokens = mask.sum(dim=1).tolist()

            for span, correct, tokens in zip(
                span_lengths, per_example_correct, per_example_tokens
            ):
                entry = totals.setdefault(span, {"correct": 0.0, "tokens": 0.0, "examples": 0.0})
                entry["correct"] += float(correct)
                entry["tokens"] += float(tokens)
                entry["examples"] += 1.0

    rows: list[dict[str, float]] = []
    for span in sorted(totals):
        entry = totals[span]
        rows.append(
            {
                "mask_length": int(span),
                "examples": int(entry["examples"]),
                "masked_tokens": int(entry["tokens"]),
                "token_accuracy": entry["correct"] / max(entry["tokens"], 1.0),
            }
        )
    return pd.DataFrame(rows)


def save_checkpoint(
    path: str | Path,
    model: nn.Module,
    config: TrainConfig,
    vocab: dict[str, Any],
    epoch: int,
    best_val_loss: float,
    history: dict[str, list[float]],
    test_metrics: dict[str, float] | None = None,
) -> Path:
    """Save weights, vocabulary and training metadata in one file."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    payload: dict[str, Any] = {
        "format_version": 1,
        "model_state_dict": model.state_dict(),
        "model_config": {
            "vocab_size": len(vocab["token_to_id"]),
            "embed_dim": config.embed_dim,
            "hidden_dim": config.hidden_dim,
            "num_layers": config.num_layers,
            "dropout": config.dropout,
            "pad_idx": PAD_ID,
        },
        "train_config": config.to_dict(),
        "vocab": vocab,
        "epoch": int(epoch),
        "best_val_loss": float(best_val_loss),
        "history": copy.deepcopy(history),
    }
    if test_metrics is not None:
        payload["test_metrics"] = test_metrics
    torch.save(payload, target)
    return target


def plot_training_curves(
    history: dict[str, list[float]],
    output_dir: str | Path = PLOT_DIR,
) -> dict[str, Path]:
    """Save the loss and accuracy curves required by the report."""
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    epochs = history["epoch"]

    loss_figure, loss_axis = plt.subplots(figsize=(8, 5))
    loss_axis.plot(
        epochs, history["train_loss"], marker="o", linewidth=2, label="Train loss"
    )
    loss_axis.plot(
        epochs, history["val_loss"], marker="s", linewidth=2, label="Validation loss"
    )
    loss_axis.set_xlabel("Epoch")
    loss_axis.set_ylabel("Cross-entropy loss")
    loss_axis.set_title("SeqLigase BiLSTM — Loss Curve")
    loss_axis.grid(alpha=0.3)
    loss_axis.legend()
    loss_figure.tight_layout()
    loss_path = destination / "loss_curve.png"
    loss_figure.savefig(loss_path, dpi=200, bbox_inches="tight")
    plt.close(loss_figure)

    accuracy_figure, accuracy_axis = plt.subplots(figsize=(8, 5))
    accuracy_axis.plot(
        epochs,
        history["train_token_accuracy"],
        marker="o",
        linewidth=2,
        label="Train token accuracy",
    )
    accuracy_axis.plot(
        epochs,
        history["val_token_accuracy"],
        marker="s",
        linewidth=2,
        label="Validation token accuracy",
    )
    accuracy_axis.plot(
        epochs,
        history["val_top3_accuracy"],
        linestyle="--",
        linewidth=1.5,
        label="Validation top-3 accuracy",
    )
    accuracy_axis.set_xlabel("Epoch")
    accuracy_axis.set_ylabel("Accuracy")
    accuracy_axis.set_ylim(0.0, 1.0)
    accuracy_axis.set_title("SeqLigase BiLSTM — Accuracy Curve")
    accuracy_axis.grid(alpha=0.3)
    accuracy_axis.legend()
    accuracy_figure.tight_layout()
    accuracy_path = destination / "accuracy_curve.png"
    accuracy_figure.savefig(accuracy_path, dpi=200, bbox_inches="tight")
    plt.close(accuracy_figure)

    return {"loss_curve": loss_path, "accuracy_curve": accuracy_path}


def summary_dataframe(metrics: dict[str, float]) -> pd.DataFrame:
    """Build the final evaluation summary table."""
    rows = [
        ("Test Loss", f"{metrics['loss']:.4f}"),
        ("Test Accuracy (%)", f"{metrics['token_accuracy'] * 100:.2f}"),
        ("Top-1 Token Accuracy (%)", f"{metrics['token_accuracy'] * 100:.2f}"),
        ("Top-3 Token Accuracy (%)", f"{metrics['top3_accuracy'] * 100:.2f}"),
        ("Character-Level Accuracy (%)", f"{metrics['char_accuracy'] * 100:.2f}"),
        ("Token Precision (macro)", f"{metrics['token_precision']:.4f}"),
        (
            "Structural Reconstruction Pass Rate (%)",
            f"{metrics['structural_pass_rate'] * 100:.2f}",
        ),
        ("RDKit Validity Rate (%)", f"{metrics['validity_rate'] * 100:.2f}"),
        ("Evaluated Sequences", f"{int(metrics['n_sequences'])}"),
        ("Evaluated Masked Tokens", f"{int(metrics['n_tokens'])}"),
    ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


def print_test_summary(metrics: dict[str, float]) -> pd.DataFrame:
    """Print and return the evaluation summary table."""
    summary = summary_dataframe(metrics)
    print("\n" + "=" * 52)
    print("Final Test Evaluation — SeqLigase BiLSTM")
    print("=" * 52)
    print(summary.to_string(index=False))
    print("=" * 52 + "\n")
    return summary
# ---------------------------------------------------------------------------
# Data preparation / training entry points
# ---------------------------------------------------------------------------
def prepare_data(
    config: TrainConfig | None = None,
    data_dir: str | Path | None = None,
    device: torch.device | None = None,
) -> dict[str, Any]:
    """Download (if needed), split, tokenise, and wrap the dataset."""
    config = config or TrainConfig()
    device = device or resolve_device()
    ensure_project_dirs()
    set_seed(config.seed)

    data_root = Path(data_dir) if data_dir is not None else RAW_DATA_DIR
    download_kaggle_dataset(data_root)

    frame = load_smiles_dataframe(data_root)
    frame = filter_smiles_dataframe(frame, max_len=config.max_len)
    if frame.empty:
        raise RuntimeError("No SMILES remain after length filtering.")

    splits = split_dataframe(frame, seed=config.seed, use_provided_split=True)

    if config.max_samples > 0:
        caps = {
            "train": config.max_samples,
            "val": max(config.max_samples // 4, 200),
            "test": max(config.max_samples // 4, 200),
        }
        splits = {
            name: part.sample(
                n=min(len(part), caps[name]), random_state=config.seed
            ).reset_index(drop=True)
            for name, part in splits.items()
        }

    print(
        "[data] Split sizes — "
        + ", ".join(f"{name}: {len(part):,}" for name, part in splits.items())
    )

    vocab = load_or_build_vocab(
        splits["train"]["smiles"].tolist(),
        VOCAB_PATH,
        force=config.rebuild_vocab,
    )
    print(f"[data] Vocabulary size: {len(vocab['token_to_id']):,} tokens")

    datasets = {
        "train": MaskedSMILESDataset(
            splits["train"]["smiles"].tolist(),
            vocab,
            max_len=config.max_len,
            max_mask_span=config.max_mask_span,
            mask_span_power=config.mask_span_power,
            seed=config.seed,
            dynamic=True,
        ),
        "val": MaskedSMILESDataset(
            splits["val"]["smiles"].tolist(),
            vocab,
            max_len=config.max_len,
            max_mask_span=config.max_mask_span,
            mask_span_power=config.mask_span_power,
            seed=config.seed,
            dynamic=False,
        ),
        "test": MaskedSMILESDataset(
            splits["test"]["smiles"].tolist(),
            vocab,
            max_len=config.max_len,
            max_mask_span=config.max_mask_span,
            mask_span_power=config.mask_span_power,
            seed=config.seed,
            dynamic=False,
        ),
    }

    pin_memory = device.type == "cuda"
    common = {
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "collate_fn": collate_masked_batch,
        "pin_memory": pin_memory,
        "persistent_workers": config.num_workers > 0,
    }
    loaders = {
        "train": DataLoader(datasets["train"], shuffle=True, **common),
        "val": DataLoader(datasets["val"], shuffle=False, **common),
        "test": DataLoader(datasets["test"], shuffle=False, **common),
    }

    return {
        "splits": splits,
        "vocab": vocab,
        "datasets": datasets,
        "loaders": loaders,
    }


def fit(
    config: TrainConfig | None = None,
    data_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Train the BiLSTM, save plots/checkpoints, and evaluate on the test set."""
    config = config or TrainConfig()
    ensure_project_dirs()
    set_seed(config.seed)

    device = resolve_device()
    use_amp = bool(config.amp and device.type == "cuda")
    print(f"[device] {device} | AMP: {use_amp}")

    prepared = prepare_data(config, data_dir=data_dir, device=device)
    vocab = prepared["vocab"]
    id_to_token: list[str] = vocab["id_to_token"]

    model = SeqLigaseBiLSTM(
        vocab_size=len(vocab["token_to_id"]),
        embed_dim=config.embed_dim,
        hidden_dim=config.hidden_dim,
        num_layers=config.num_layers,
        dropout=config.dropout,
        pad_idx=PAD_ID,
    ).to(device)
    print(f"[model] Trainable parameters: {model.num_parameters:,}")

    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)  # PAD/non-masked = -100
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=config.scheduler_factor,
        patience=config.scheduler_patience,
        min_lr=1e-6,
    )
    scaler = create_grad_scaler(use_amp, device.type)

    history: dict[str, list[float]] = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "train_token_accuracy": [],
        "val_token_accuracy": [],
        "train_top3_accuracy": [],
        "val_top3_accuracy": [],
        "train_char_accuracy": [],
        "val_char_accuracy": [],
        "train_token_precision": [],
        "val_token_precision": [],
        "val_structural_pass_rate": [],
        "val_validity_rate": [],
        "lr": [],
    }

    best_val_loss = math.inf
    epochs_without_improvement = 0
    best_epoch = 0
    best_path = CHECKPOINT_DIR / config.checkpoint_name
    last_path = CHECKPOINT_DIR / LAST_CHECKPOINT_NAME

    for epoch in range(1, config.epochs + 1):
        prepared["datasets"]["train"].set_epoch(epoch)
        started_at = time.time()

        train_metrics = train_epoch(
            model=model,
            loader=prepared["loaders"]["train"],
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            config=config,
            use_amp=use_amp,
            id_to_token=id_to_token,
        )
        val_metrics = evaluate(
            model=model,
            loader=prepared["loaders"]["val"],
            criterion=criterion,
            device=device,
            use_amp=use_amp,
            id_to_token=id_to_token,
            check_rdkit=True,
        )
        scheduler.step(val_metrics["loss"])
        current_lr = float(optimizer.param_groups[0]["lr"])
        elapsed = time.time() - started_at

        history["epoch"].append(epoch)
        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_token_accuracy"].append(train_metrics["token_accuracy"])
        history["val_token_accuracy"].append(val_metrics["token_accuracy"])
        history["train_top3_accuracy"].append(train_metrics["top3_accuracy"])
        history["val_top3_accuracy"].append(val_metrics["top3_accuracy"])
        history["train_char_accuracy"].append(train_metrics["char_accuracy"])
        history["val_char_accuracy"].append(val_metrics["char_accuracy"])
        history["train_token_precision"].append(train_metrics["token_precision"])
        history["val_token_precision"].append(val_metrics["token_precision"])
        history["val_structural_pass_rate"].append(
            val_metrics["structural_pass_rate"]
        )
        history["val_validity_rate"].append(val_metrics["validity_rate"])
        history["lr"].append(current_lr)

        improved = val_metrics["loss"] < best_val_loss - 1e-6
        if improved:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            epochs_without_improvement = 0
            save_checkpoint(
                best_path,
                model,
                config,
                vocab,
                epoch,
                best_val_loss,
                history,
            )
        else:
            epochs_without_improvement += 1

        save_checkpoint(
            last_path,
            model,
            config,
            vocab,
            epoch,
            best_val_loss,
            history,
        )

        marker = " *best*" if improved else ""
        print(
            f"[epoch {epoch:02d}/{config.epochs:02d}] "
            f"loss {train_metrics['loss']:.4f}/{val_metrics['loss']:.4f} | "
            f"acc {train_metrics['token_accuracy'] * 100:.2f}/"
            f"{val_metrics['token_accuracy'] * 100:.2f} | "
            f"top3 {val_metrics['top3_accuracy'] * 100:.2f} | "
            f"char {val_metrics['char_accuracy'] * 100:.2f} | "
            f"prec {val_metrics['token_precision']:.3f} | "
            f"struct {val_metrics['structural_pass_rate'] * 100:.2f} | "
            f"lr {current_lr:.2e} | {elapsed:.1f}s{marker}"
        )

        if epochs_without_improvement >= config.early_stopping_patience:
            print(
                f"[early-stop] No validation improvement for "
                f"{config.early_stopping_patience} epochs."
            )
            break

    print(f"[train] Best epoch: {best_epoch} | best val loss: {best_val_loss:.4f}")

    best_model, best_checkpoint = SeqLigaseBiLSTM.from_checkpoint(
        best_path, map_location=device
    )
    best_model.to(device)

    test_metrics = evaluate(
        model=best_model,
        loader=prepared["loaders"]["test"],
        criterion=criterion,
        device=device,
        use_amp=use_amp,
        id_to_token=id_to_token,
        check_rdkit=True,
    )
    print_test_summary(test_metrics)

    span_breakdown = analyze_by_span_length(
        model=best_model,
        loader=prepared["loaders"]["test"],
        criterion=criterion,
        device=device,
        use_amp=use_amp,
    )
    if not span_breakdown.empty:
        print("Token accuracy by masked-span length:")
        print(span_breakdown.to_string(index=False))
    span_breakdown.to_csv(ARTIFACT_DIR / "span_length_metrics.csv", index=False)

    plot_paths = plot_training_curves(history, PLOT_DIR)

    best_checkpoint["test_metrics"] = test_metrics
    best_checkpoint["span_breakdown"] = span_breakdown.to_dict(orient="records")
    best_checkpoint["history"] = copy.deepcopy(history)
    torch.save(best_checkpoint, best_path)

    (ARTIFACT_DIR / "history.json").write_text(
        json.dumps(history, indent=2), encoding="utf-8"
    )
    (ARTIFACT_DIR / "test_metrics.json").write_text(
        json.dumps(test_metrics, indent=2), encoding="utf-8"
    )

    return {
        "history": history,
        "test_metrics": test_metrics,
        "span_breakdown": span_breakdown,
        "checkpoint": str(best_path),
        "plots": {name: str(path) for name, path in plot_paths.items()},
        "vocab_size": len(vocab["token_to_id"]),
    }


def evaluate_checkpoint(
    config: TrainConfig | None = None,
    checkpoint_path: str | Path | None = None,
    data_dir: str | Path | None = None,
) -> dict[str, float]:
    """Reload the best checkpoint and evaluate it on the held-out test split."""
    config = config or TrainConfig()
    device = resolve_device()
    use_amp = bool(config.amp and device.type == "cuda")
    prepared = prepare_data(config, data_dir=data_dir, device=device)

    path = (
        Path(checkpoint_path)
        if checkpoint_path is not None
        else CHECKPOINT_DIR / config.checkpoint_name
    )
    model, _ = SeqLigaseBiLSTM.from_checkpoint(path, map_location=device)
    model.to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)  # PAD/non-masked = -100
    metrics = evaluate(
        model=model,
        loader=prepared["loaders"]["test"],
        criterion=criterion,
        device=device,
        use_amp=use_amp,
        id_to_token=prepared["vocab"]["id_to_token"],
        check_rdkit=True,
    )
    print_test_summary(metrics)

    ensure_project_dirs()
    (ARTIFACT_DIR / "test_metrics.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )
    return metrics


def main() -> None:
    """Command-line entry point for training."""
    defaults = TrainConfig()
    parser = argparse.ArgumentParser(
        description="Train the SeqLigase BiLSTM SMILES-infilling model."
    )
    parser.add_argument("--epochs", type=int, default=defaults.epochs)
    parser.add_argument("--batch-size", type=int, default=defaults.batch_size)
    parser.add_argument("--lr", type=float, default=defaults.lr)
    parser.add_argument("--embed-dim", type=int, default=defaults.embed_dim)
    parser.add_argument("--hidden-dim", type=int, default=defaults.hidden_dim)
    parser.add_argument("--num-layers", type=int, default=defaults.num_layers)
    parser.add_argument("--num-workers", type=int, default=defaults.num_workers)
    parser.add_argument("--max-samples", type=int, default=defaults.max_samples)
    parser.add_argument("--max-mask-span", type=int, default=defaults.max_mask_span)
    parser.add_argument(
        "--mask-span-power", type=float, default=defaults.mask_span_power
    )
    parser.add_argument("--data-dir", type=Path, default=None)
    parser.add_argument("--rebuild-vocab", action="store_true")
    parser.add_argument("--no-amp", action="store_true")
    arguments = parser.parse_args()

    config = TrainConfig(
        epochs=arguments.epochs,
        batch_size=arguments.batch_size,
        lr=arguments.lr,
        embed_dim=arguments.embed_dim,
        hidden_dim=arguments.hidden_dim,
        num_layers=arguments.num_layers,
        num_workers=arguments.num_workers,
        max_samples=arguments.max_samples,
        max_mask_span=arguments.max_mask_span,
        mask_span_power=arguments.mask_span_power,
        rebuild_vocab=arguments.rebuild_vocab,
        amp=not arguments.no_amp,
    )
    result = fit(config, data_dir=arguments.data_dir)
    print(f"[done] Best checkpoint: {result['checkpoint']}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile predict.py
"""Interactive inference and OpenRouter-ready prediction wrapper."""

from __future__ import annotations

import argparse
import json
import math
from pathlib import Path
from typing import Any

import torch
import torch.nn.functional as functional

from config import (
    BEST_CHECKPOINT_NAME,
    CHECKPOINT_DIR,
    EOS_ID,
    EOS_TOKEN,
    MASK_ID,
    MASK_TOKEN,
    PAD_ID,
    SOS_ID,
    SOS_TOKEN,
    UNK_ID,
    UNK_TOKEN,
)
from data_utils import canonicalize_smiles, detokenize_smiles, tokenize_smiles
from models.bilstm import SeqLigaseBiLSTM

_MODEL_CACHE: dict[tuple[str, str], tuple[Any, dict[str, Any], torch.device, dict]] = {}


def _resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None:
        if torch.backends.mps.is_available():
            return torch.device("mps")
    return torch.device("cpu")


def load_model(
    checkpoint_path: str | Path | None = None,
    device: torch.device | None = None,
    use_cache: bool = True,
) -> tuple[SeqLigaseBiLSTM, dict[str, Any], torch.device, dict[str, Any]]:
    """Load the trained checkpoint (cached by path and device)."""
    path = (
        Path(checkpoint_path)
        if checkpoint_path is not None
        else CHECKPOINT_DIR / BEST_CHECKPOINT_NAME
    )
    device = device or _resolve_device()
    cache_key = (str(path.resolve()), str(device))
    if use_cache and cache_key in _MODEL_CACHE:
        return _MODEL_CACHE[cache_key]

    model, checkpoint = SeqLigaseBiLSTM.from_checkpoint(path, map_location=device)
    model.to(device)
    model.eval()

    vocab = checkpoint["vocab"]
    id_to_token = vocab.get("id_to_token", [])
    if isinstance(id_to_token, dict):
        id_to_token = [id_to_token[str(index)] for index in range(len(id_to_token))]
    vocab["id_to_token"] = id_to_token

    payload = (model, vocab, device, checkpoint)
    if use_cache:
        _MODEL_CACHE[cache_key] = payload
    return payload


def _normalise_placeholders(masked_smiles: str) -> str:
    text = masked_smiles.strip()
    text = text.replace("<MASK>", "[MASK]").replace("<mask>", "[MASK]")
    return text


def _tokenize_with_masks(masked_smiles: str) -> tuple[list[str], int, int]:
    """Return clean tokens, the first mask index, and the placeholder count."""
    tokens = tokenize_smiles(_normalise_placeholders(masked_smiles))
    positions: list[int] = []
    clean_tokens: list[str] = []

    for token in tokens:
        if token.lower() in {"[mask]", "*", "[*]"}:
            positions.append(len(clean_tokens))
            clean_tokens.append(MASK_TOKEN)
        else:
            clean_tokens.append(token)

    if not positions:
        raise ValueError(
            f"No mask placeholder found in {masked_smiles!r}. Use [MASK], "
            "<MASK>, or * to mark the fragment to infill."
        )

    expected = list(range(positions[0], positions[0] + len(positions)))
    if positions != expected:
        raise ValueError(
            "Only one contiguous masked fragment is supported. "
            f"Found mask positions {positions}."
        )

    return clean_tokens, positions[0], len(positions)


def _beam_decode(
    log_probabilities: torch.Tensor,
    beam_width: int,
) -> list[tuple[float, list[int]]]:
    """Beam-search a set of per-position log distributions."""
    beams: list[tuple[float, list[int]]] = [(0.0, [])]
    for position in range(log_probabilities.size(0)):
        top_values, top_indices = log_probabilities[position].topk(
            min(beam_width, log_probabilities.size(-1))
        )
        expanded: list[tuple[float, list[int]]] = []
        for score, sequence in beams:
            for log_probability, token_id in zip(
                top_values.tolist(), top_indices.tolist()
            ):
                if not math.isfinite(log_probability):
                    continue
                expanded.append((score + log_probability, sequence + [token_id]))
        expanded.sort(key=lambda item: item[0], reverse=True)
        beams = expanded[:beam_width]
        if not beams:
            break
    return beams


def infill_candidates(
    masked_smiles: str,
    model: SeqLigaseBiLSTM,
    vocab: dict[str, Any],
    device: torch.device,
    top_k: int = 3,
    max_infill_len: int = 12,
    beam_width: int | None = None,
) -> list[dict[str, Any]]:
    """Enumerate fragment lengths, beam-decode each, and rank the candidates."""
    if top_k < 1:
        raise ValueError("top_k must be >= 1")

    token_to_id: dict[str, int] = vocab["token_to_id"]
    id_to_token: list[str] = vocab["id_to_token"]

    clean_tokens, mask_start, placeholder_count = _tokenize_with_masks(masked_smiles)
    prefix_tokens = clean_tokens[:mask_start]
    suffix_tokens = clean_tokens[mask_start + placeholder_count :]

    if placeholder_count > 1:
        lengths = [placeholder_count]
    else:
        lengths = list(range(1, int(max_infill_len) + 1))

    beam_width = beam_width or max(top_k * 4, 16)
    special_ids = torch.tensor(
        [PAD_ID, UNK_ID, MASK_ID, SOS_ID, EOS_ID], device=device, dtype=torch.long
    )

    candidates: list[dict[str, Any]] = []
    model.eval()

    for length in lengths:
        input_tokens = (
            [SOS_TOKEN]
            + prefix_tokens
            + [MASK_TOKEN] * length
            + suffix_tokens
            + [EOS_TOKEN]
        )
        input_ids = torch.tensor(
            [[token_to_id.get(token, UNK_ID) for token in input_tokens]],
            device=device,
            dtype=torch.long,
        )
        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            logits = model(input_ids, attention_mask)

        start_index = len(prefix_tokens) + 1
        position_logits = logits[0, start_index : start_index + length, :].float()
        log_probabilities = functional.log_softmax(position_logits, dim=-1)
        log_probabilities[:, special_ids] = float("-inf")
        log_probabilities = log_probabilities - torch.logsumexp(
            log_probabilities, dim=-1, keepdim=True
        )
        probabilities = log_probabilities.exp()
        finite_log_probabilities = torch.where(
            torch.isfinite(log_probabilities),
            log_probabilities,
            torch.zeros_like(log_probabilities),
        )
        entropy_nats = float(
            (-(probabilities * finite_log_probabilities).sum(dim=-1)).mean()
        )

        for sum_log_probability, fragment_ids in _beam_decode(
            log_probabilities, beam_width
        ):
            fragment_tokens = [
                id_to_token[token_id]
                if 0 <= token_id < len(id_to_token)
                else UNK_TOKEN
                for token_id in fragment_ids
            ]
            fragment = detokenize_smiles(fragment_tokens)
            full_smiles = detokenize_smiles(
                prefix_tokens + fragment_tokens + suffix_tokens
            )
            canonical_smiles = canonicalize_smiles(full_smiles)
            mean_log_probability = sum_log_probability / max(length, 1)
            confidence = math.exp(mean_log_probability)
            probability = math.exp(sum_log_probability)

            candidates.append(
                {
                    "masked_smiles": masked_smiles,
                    "fragment": fragment,
                    "full_smiles": full_smiles,
                    "probability_percent": probability * 100.0,
                    "confidence_score": confidence,
                    "confidence_percent": confidence * 100.0,
                    "uncertainty_entropy_nats": entropy_nats,
                    "uncertainty_entropy_bits": entropy_nats / math.log(2.0),
                    "rdkit_valid": canonical_smiles is not None,
                    "canonical_smiles": canonical_smiles,
                    "mask_length": int(length),
                    "mean_log_probability": float(mean_log_probability),
                }
            )

    deduplicated: dict[str, dict[str, Any]] = {}
    for candidate in candidates:
        key = candidate["full_smiles"]
        existing = deduplicated.get(key)
        if (
            existing is None
            or candidate["confidence_score"] > existing["confidence_score"]
        ):
            deduplicated[key] = candidate

    ranked = sorted(
        deduplicated.values(),
        key=lambda item: (
            bool(item["rdkit_valid"]),
            item["probability_percent"],
            item["confidence_score"],
        ),
        reverse=True,
    )[:top_k]

    for rank, candidate in enumerate(ranked, start=1):
        candidate["rank"] = rank
    return ranked
def get_bilstm_predictions(
    masked_smiles: str,
    top_k: int = 3,
    checkpoint_path: str | Path | None = None,
    max_infill_len: int | None = None,
) -> list[dict[str, Any]]:
    """Return JSON-serialisable infilling candidates for an orchestrator.

    Each item in the returned list has this schema::

        {
            "masked_smiles": str,
            "fragment": str,
            "full_smiles": str,
            "probability_percent": float,
            "confidence_score": float,       # 0..1, length-normalised
            "confidence_percent": float,
            "uncertainty_entropy_nats": float,
            "uncertainty_entropy_bits": float,
            "rdkit_valid": bool,
            "canonical_smiles": str | None,
            "mask_length": int,
            "mean_log_probability": float,
            "rank": int
        }

    Candidates are sorted with RDKit-valid structures first, then by raw
    joint probability (the probability of the exact reconstruction).

    Parameters
    ----------
    masked_smiles:
        SMILES containing one contiguous ``[MASK]`` (or ``*``) fragment.
    top_k:
        Number of candidates to return.
    checkpoint_path:
        Optional path to a checkpoint. Defaults to
        ``checkpoints/seqligase_bilstm_best.pt``.
    max_infill_len:
        Maximum fragment length to enumerate. Defaults to the training
        ``max_mask_span`` stored in the checkpoint.
    """
    model, vocab, device, checkpoint = load_model(checkpoint_path)
    train_config = checkpoint.get("train_config", {})
    configured_limit = int(train_config.get("max_mask_span", 12))
    limit = int(max_infill_len) if max_infill_len is not None else configured_limit

    return infill_candidates(
        masked_smiles=masked_smiles,
        model=model,
        vocab=vocab,
        device=device,
        top_k=top_k,
        max_infill_len=limit,
    )


def print_candidates(candidates: list[dict[str, Any]]) -> None:
    """Pretty-print candidates in the required CLI format."""
    if not candidates:
        print("No candidates were produced.")
        return
    for candidate in candidates:
        print(f"Candidate {candidate['rank']}")
        print(f"  Predicted Infilled Fragment : {candidate['fragment']}")
        print(f"  Full Reconstructed SMILES   : {candidate['full_smiles']}")
        print(
            "  Raw Model Probability (%)   : "
            f"{candidate['probability_percent']:.4f}"
        )
        print(
            "  Confidence Score (per-token): "
            f"{candidate['confidence_score']:.4f} "
            f"({candidate['confidence_percent']:.2f}%)"
        )
        print(f"  RDKit Syntax Validity       : {candidate['rdkit_valid']}")
        print(
            "  Uncertainty / Entropy       : "
            f"{candidate['uncertainty_entropy_nats']:.4f} nats "
            f"({candidate['uncertainty_entropy_bits']:.4f} bits)"
        )
        if candidate["canonical_smiles"] is not None:
            print(f"  Canonical SMILES            : {candidate['canonical_smiles']}")
        print()


def interactive_cli(
    top_k: int = 3,
    checkpoint_path: str | Path | None = None,
    max_infill_len: int | None = None,
) -> None:
    """Run a read-eval-print loop for masked SMILES."""
    print("=" * 72)
    print("SeqLigase BiLSTM — interactive SMILES infilling")
    print("Enter a masked SMILES such as c1cc([MASK])ccc1C(=O)O")
    print("One [MASK] searches all fragment lengths; repeat it to fix the length.")
    print("Type 'quit' to exit.")
    print("=" * 72)

    while True:
        try:
            masked_smiles = input("\nMasked SMILES> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye.")
            break
        if masked_smiles.lower() in {"quit", "exit", "q"}:
            print("Goodbye.")
            break
        if not masked_smiles:
            continue
        try:
            candidates = get_bilstm_predictions(
                masked_smiles,
                top_k=top_k,
                checkpoint_path=checkpoint_path,
                max_infill_len=max_infill_len,
            )
        except Exception as exc:
            print(f"[error] {exc}")
            continue
        print()
        print_candidates(candidates)


def main() -> None:
    """Command-line entry point for inference."""
    parser = argparse.ArgumentParser(
        description="Predict masked SMILES fragments with the SeqLigase BiLSTM."
    )
    parser.add_argument(
        "--masked-smiles",
        type=str,
        default=None,
        help="Masked SMILES. If omitted, starts an interactive prompt.",
    )
    parser.add_argument("--top-k", type=int, default=3)
    parser.add_argument(
        "--checkpoint",
        type=Path,
        default=CHECKPOINT_DIR / BEST_CHECKPOINT_NAME,
    )
    parser.add_argument("--max-infill-len", type=int, default=None)
    parser.add_argument(
        "--json",
        action="store_true",
        help="Print the JSON payload instead of the formatted report.",
    )
    arguments = parser.parse_args()

    if arguments.masked_smiles is None:
        interactive_cli(
            top_k=arguments.top_k,
            checkpoint_path=arguments.checkpoint,
            max_infill_len=arguments.max_infill_len,
        )
        return

    candidates = get_bilstm_predictions(
        arguments.masked_smiles,
        top_k=arguments.top_k,
        checkpoint_path=arguments.checkpoint,
        max_infill_len=arguments.max_infill_len,
    )
    if arguments.json:
        print(json.dumps(candidates, indent=2, ensure_ascii=False))
    else:
        print_candidates(candidates)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_model.py
"""Package a SeqLigase checkpoint for external pipelines and Kaggle download."""

from __future__ import annotations

import argparse
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import torch

from config import (
    BEST_CHECKPOINT_NAME,
    CHECKPOINT_DIR,
    EXPORT_DIR,
    PROJECT_ROOT,
)
from models.bilstm import SeqLigaseBiLSTM


def _load_checkpoint(checkpoint_path: str | Path) -> dict[str, Any]:
    path = Path(checkpoint_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {path}. Train the model first."
        )
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:  # torch < 1.13
        return torch.load(path, map_location="cpu")


def export_model(
    checkpoint_path: str | Path | None = None,
    export_dir: str | Path = EXPORT_DIR,
    create_zip: bool = True,
) -> dict[str, Path]:
    """Bundle weights, config, and vocabulary into portable artefacts.

    Returns a dictionary with ``weights``, ``vocab``, ``model_card``, and
    (when requested) ``zip`` paths.
    """
    source = (
        Path(checkpoint_path)
        if checkpoint_path is not None
        else CHECKPOINT_DIR / BEST_CHECKPOINT_NAME
    )
    checkpoint = _load_checkpoint(source)

    destination = Path(export_dir)
    destination.mkdir(parents=True, exist_ok=True)

    payload: dict[str, Any] = {
        "format_version": 1,
        "exported_at": datetime.now(timezone.utc).isoformat(),
        "source_checkpoint": str(source),
        "model_state_dict": checkpoint["model_state_dict"],
        "model_config": checkpoint["model_config"],
        "train_config": checkpoint.get("train_config", {}),
        "vocab": checkpoint["vocab"],
        "history": checkpoint.get("history", {}),
        "test_metrics": checkpoint.get("test_metrics", {}),
    }

    weights_path = destination / "seqligase_bilstm_export.pt"
    torch.save(payload, weights_path)

    vocab_path = destination / "vocab.json"
    vocab_path.write_text(
        json.dumps(checkpoint["vocab"], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    metrics_text = json.dumps(
        checkpoint.get("test_metrics", {}), indent=2, ensure_ascii=False
    )
    model_card_path = destination / "MODEL_CARD.md"
    model_card_path.write_text(
        "# SeqLigase BiLSTM — exported model\n\n"
        f"- Source checkpoint: `{source}`\n"
        f"- Exported (UTC): {payload['exported_at']}\n"
        f"- Vocabulary size: {len(checkpoint['vocab']['token_to_id'])}\n"
        f"- Model config: `{json.dumps(checkpoint['model_config'])}`\n\n"
        "## Test metrics\n\n"
        f"```json\n{metrics_text}\n```\n\n"
        "## Loading\n\n"
        "```python\n"
        "from models.bilstm import SeqLigaseBiLSTM\n"
        "model, bundle = SeqLigaseBiLSTM.from_checkpoint("
        "'artifacts/exports/seqligase_bilstm_export.pt')\n"
        "model.eval()\n"
        "```\n",
        encoding="utf-8",
    )

    result = {
        "weights": weights_path,
        "vocab": vocab_path,
        "model_card": model_card_path,
    }

    if create_zip:
        zip_path = destination / "seqligase_bilstm_export.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
            for file_path in (weights_path, vocab_path, model_card_path):
                archive.write(file_path, arcname=file_path.name)
        result["zip"] = zip_path

    return result


def load_exported_model(
    export_path: str | Path,
    map_location: str | torch.device = "cpu",
) -> tuple[SeqLigaseBiLSTM, dict[str, Any], dict[str, Any]]:
    """Load an export bundle and return ``(model, vocab, metadata)``."""
    bundle = _load_checkpoint(export_path)
    model = SeqLigaseBiLSTM(**bundle["model_config"])
    model.load_state_dict(bundle["model_state_dict"])
    model.to(map_location)
    model.eval()
    return model, bundle["vocab"], bundle


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Export the trained SeqLigase checkpoint for external use."
    )
    parser.add_argument(
        "--checkpoint",
        type=Path,
        default=CHECKPOINT_DIR / BEST_CHECKPOINT_NAME,
    )
    parser.add_argument("--export-dir", type=Path, default=EXPORT_DIR)
    parser.add_argument("--no-zip", action="store_true")
    arguments = parser.parse_args()

    paths = export_model(
        checkpoint_path=arguments.checkpoint,
        export_dir=arguments.export_dir,
        create_zip=not arguments.no_zip,
    )
    print("Export complete:")
    for name, path in paths.items():
        size_mb = path.stat().st_size / (1024 * 1024)
        try:
            display_path = path.relative_to(PROJECT_ROOT)
        except ValueError:
            display_path = path
        print(f"  {name:12s} {display_path} ({size_mb:.2f} MB)")


if __name__ == "__main__":
    main()


### Project metadata files

`requirements.txt` pins the runtime dependencies and `.gitignore` keeps the 41 MB dataset, checkpoints, and artefacts out of version control.


In [ ]:
from pathlib import Path

REQUIREMENTS = """torch>=2.1.0
rdkit>=2023.9.1
matplotlib>=3.7.0
pandas>=2.0.0
kagglehub>=0.3.0
scikit-learn>=1.3.0
numpy>=1.24.0
"""

GITIGNORE = """# ---------------------------------------------------------------------------
# Dataset / generated artefacts
# ---------------------------------------------------------------------------
data/raw/*
!data/raw/.gitkeep
checkpoints/*
!checkpoints/.gitkeep
artifacts/plots/*
!artifacts/plots/.gitkeep
artifacts/exports/*
!artifacts/exports/.gitkeep
artifacts/*.json

# ---------------------------------------------------------------------------
# Python
# ---------------------------------------------------------------------------
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
build/
dist/
*.egg-info/
.eggs/

# ---------------------------------------------------------------------------
# Environments / secrets
# ---------------------------------------------------------------------------
.env
.env.*
.venv/
venv/
env/
kaggle.json
.kaggle/

# ---------------------------------------------------------------------------
# Notebooks / editors / OS
# ---------------------------------------------------------------------------
.ipynb_checkpoints/
.vscode/
.idea/
.DS_Store
Thumbs.db

# ---------------------------------------------------------------------------
# Large exported weights
# ---------------------------------------------------------------------------
*.pt
*.pth
*.ckpt
*.zip
"""

Path("requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
Path(".gitignore").write_text(GITIGNORE, encoding="utf-8")

print("Wrote requirements.txt and .gitignore")
print(REQUIREMENTS)


## 3. Download and inspect the dataset

`download_kaggle_dataset()` resolves data in this order:

1. Existing CSVs in `data/raw/`
2. A dataset attached under `/kaggle/input` (the recommended Kaggle path)
3. `kagglehub.dataset_download(...)`
4. The `kaggle datasets download` CLI

Only CSV/JSON files are copied from the 193 MB Kaggle bundle; the large ECFP `.npy` arrays are not needed for this task.


In [ ]:
from config import RAW_DATA_DIR
from data_utils import discover_csv_files, download_kaggle_dataset

DATA_ROOT = download_kaggle_dataset(RAW_DATA_DIR)

print("Dataset root:", DATA_ROOT)
print("Available tabular files:")
for path in discover_csv_files(DATA_ROOT):
    size_mb = path.stat().st_size / 1e6
    print(f"  - {path.relative_to(DATA_ROOT)}  ({size_mb:.1f} MB)")


## 4. Load SMILES, inspect lengths, and create the split

The dataset already provides a leakage-checked 80/10/10 split. `split_dataframe()` reuses that split when the `split` column exists and otherwise performs a seeded two-stage `sklearn` split with `random_state=42`.


In [ ]:
import pandas as pd

from data_utils import (
    filter_smiles_dataframe,
    load_smiles_dataframe,
    split_dataframe,
    tokenize_smiles,
)

frame = load_smiles_dataframe(DATA_ROOT)
frame = filter_smiles_dataframe(frame, max_len=128)
print(f"Usable molecules: {len(frame):,}")

display(frame.head(8))

lengths = frame["smiles"].map(lambda value: len(tokenize_smiles(value)))
display(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame("token_length").T)

SPLITS = split_dataframe(frame, seed=42, use_provided_split=True)
for name, part in SPLITS.items():
    percentage = 100.0 * len(part) / len(frame)
    print(f"{name:>5}: {len(part):,} molecules ({percentage:.1f}%)")


## 5. Build the token vocabulary

The tokenizer keeps bracketed atoms (`[C@@H]`), two-letter elements (`Cl`, `Br`), chirality markers, and ring-closure percentages as single tokens. The five special tokens occupy ids 0–4, and padding positions are labelled `-100` so `CrossEntropyLoss` ignores them.

The vocabulary is built **only from the training split** to avoid validation/test leakage, then saved to `vocab.json`.


In [ ]:
from config import VOCAB_PATH
from data_utils import load_or_build_vocab

vocab = load_or_build_vocab(
    SPLITS["train"]["smiles"].tolist(),
    VOCAB_PATH,
    force=True,
)

print(f"Vocabulary size : {len(vocab['token_to_id']):,}")
print(f"Special tokens  : {vocab['special_tokens']}")
print(f"First 50 tokens : {vocab['id_to_token'][:50]}")
print(f"Saved to        : {VOCAB_PATH}")


## 6. Dynamic masking dataset

Each epoch, `MaskedSMILESDataset` replaces a random contiguous token span with `<MASK>` and keeps the original ids as labels. The loss is computed only on those masked positions, so the model learns infilling rather than trivial copying.

- Training dataset: `dynamic=True` → new masks every epoch
- Validation/test datasets: `dynamic=False` → stable, comparable metrics
- `mask_span_power=4.0` biases span lengths toward short fragments (`span = 1 + int(max_span * u**4)`), which is the single biggest driver of token accuracy: short gaps have far more local context. Roughly 64% of masks are a single token, ~12% are two tokens, and the tail reaches 6.
- Token ids are cached once in `__init__`, so epochs no longer re-tokenise the 85k-molecule corpus.


In [ ]:
from torch.utils.data import DataLoader

from config import PAD_ID
from data_utils import MaskedSMILESDataset, collate_masked_batch

preview_train = MaskedSMILESDataset(
    SPLITS["train"]["smiles"].tolist(),
    vocab,
    max_len=128,
    max_mask_span=6,
    mask_span_power=4.0,
    seed=42,
    dynamic=True,
)
preview_val = MaskedSMILESDataset(
    SPLITS["val"]["smiles"].tolist(),
    vocab,
    max_len=128,
    max_mask_span=6,
    mask_span_power=4.0,
    seed=42,
    dynamic=False,
)
preview_test = MaskedSMILESDataset(
    SPLITS["test"]["smiles"].tolist(),
    vocab,
    max_len=128,
    max_mask_span=6,
    mask_span_power=4.0,
    seed=42,
    dynamic=False,
)

preview_loader = DataLoader(
    preview_train,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_masked_batch,
)


def render_ids(token_ids):
    return "".join(
        vocab["id_to_token"][int(token_id)]
        for token_id in token_ids
        if int(token_id) != PAD_ID
    )


sample = preview_train[0]
true_fragment = "".join(
    vocab["id_to_token"][int(token_id)]
    for token_id in sample["original_ids"][sample["mask_start"] : sample["mask_end"]]
)
print("Original        :", sample["original_smiles"])
print("Masked input    :", render_ids(sample["input_ids"]))
print("Target fragment :", true_fragment)
print("Mask span       :", int(sample["mask_start"]), "->", int(sample["mask_end"]))

batch = next(iter(preview_loader))
print("Batch input shape:", tuple(batch["input_ids"].shape))
print("Masked tokens/batch:", int(batch["labels"].ne(-100).sum()))


## 7. BiLSTM model

`SeqLigaseBiLSTM` = learned token embeddings (`embed_dim=192`) → 3-layer bidirectional LSTM (`hidden_dim=384`, so a 768-wide output) → LayerNorm → linear projection to the vocabulary. Padding is handled with `pack_padded_sequence`, and the padding embedding is fixed at zero.

The extra depth and width roughly double the parameter count versus the 128/256/2 baseline (about 5.5M trainable parameters), which raises the ceiling of the token-accuracy plateau. Output logits for `<PAD>`, `<UNK>`, `<MASK>`, `<SOS>`, and `<EOS>` are masked out (`restrict_output_logits`), so the softmax never spends probability on tokens that can never be a valid reconstruction.


In [ ]:
from config import PAD_ID
from models.bilstm import SeqLigaseBiLSTM

preview_model = SeqLigaseBiLSTM(
    vocab_size=len(vocab["token_to_id"]),
    embed_dim=192,
    hidden_dim=384,
    num_layers=3,
    dropout=0.2,
    pad_idx=PAD_ID,
)
print(preview_model)
print(f"Trainable parameters: {preview_model.num_parameters:,}")


## 8. Train

Defaults: `AdamW(lr=1e-3, weight_decay=1e-4)`, `ReduceLROnPlateau(factor=0.5, patience=3)`, gradient clipping at 1.0, mixed precision on CUDA, and early stopping after 8 epochs without validation improvement.

Every time validation loss improves, the checkpoint is written to `checkpoints/seqligase_bilstm_best.pt`. After training, the best checkpoint is reloaded and evaluated on the held-out test split.

### Optimizations versus the initial plateau (~58% val token accuracy)

| Change | Why it lifts accuracy |
|---|---|
| `max_mask_span=6` (was 12) | Long gaps are almost unguessable from context; capping at 6 removes the hardest tail of the task. |
| `mask_span_power=4.0` | Power-law span sampling puts ~64% of probability on single-token gaps and ~12% on two-token gaps, while still training on all lengths up to 6. |
| Special-token logit ban | `<PAD>`/`<SOS>`/`<EOS>`/`<UNK>`/`<MASK>` can no longer absorb probability mass or win the argmax. |
| `embed_dim=192`, `hidden_dim=384`, `num_layers=3` | More capacity to fit QM9's chemistry; ~5.5M parameters still trains in seconds per epoch on a T4. |
| Cached tokenisation + `scheduler_patience=3` | Faster epochs and a less trigger-happy LR decay, so the model is not cut off mid-improvement. |

`fit()` also prints (and saves to `artifacts/span_length_metrics.csv`) token accuracy broken down by masked-span length, so you can see exactly where the remaining errors live.

> **Quick smoke test:** replace the config below with `TrainConfig(epochs=2, max_samples=5000, num_workers=2)`.


In [ ]:
from config import TrainConfig
from train import fit

config = TrainConfig(
    seed=42,
    embed_dim=192,
    hidden_dim=384,
    num_layers=3,
    dropout=0.2,
    max_len=128,
    max_mask_span=6,
    mask_span_power=4.0,
    batch_size=256,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    grad_clip=1.0,
    scheduler_factor=0.5,
    scheduler_patience=3,
    early_stopping_patience=8,
    num_workers=2,
    amp=True,
    rebuild_vocab=False,
)

result = fit(config)
print("\nBest checkpoint:", result["checkpoint"])
print("Plot artefacts :", result["plots"])


## 9. Training curves

The curves are saved automatically by `plot_training_curves()` in `train.py`.


In [ ]:
from IPython.display import Image, display

for plot_path in (
    "artifacts/plots/loss_curve.png",
    "artifacts/plots/accuracy_curve.png",
):
    print(plot_path)
    display(Image(filename=plot_path))


## 10. Final test evaluation

Reload `checkpoints/seqligase_bilstm_best.pt` and score the held-out 10% test split. The table reports test loss, token accuracy (top-1), top-3 token accuracy, character-level accuracy, macro token precision, structural reconstruction pass rate (RDKit-valid **and** canonically equal to the original), and raw RDKit validity.


In [ ]:
import json

import pandas as pd

from config import BEST_CHECKPOINT_NAME, CHECKPOINT_DIR
from train import evaluate_checkpoint, summary_dataframe

test_metrics = evaluate_checkpoint(
    config,
    checkpoint_path=CHECKPOINT_DIR / BEST_CHECKPOINT_NAME,
)

display(summary_dataframe(test_metrics).style.hide(axis="index"))
print(json.dumps(test_metrics, indent=2))


### Accuracy by masked-span length

Aggregate token accuracy hides the difficulty curve: a 1-token gap is nearly deterministic from local context, while an 8-token gap is not. The table below is also written to `artifacts/span_length_metrics.csv`.


In [ ]:
span_breakdown = result.get("span_breakdown")
if span_breakdown is None or span_breakdown.empty:
    print("Span breakdown unavailable — re-run the training cell.")
else:
    display(
        span_breakdown.style.format(
            {"token_accuracy": "{:.2%}"}
        ).hide(axis="index")
    )


## 11. Inference demo and OpenRouter wrapper

`get_bilstm_predictions()` enumerates candidate fragment lengths, runs a beam search over the model's per-position distributions, checks every reconstruction with RDKit, and returns a JSON-serialisable ranked list. Valid structures are ranked ahead of invalid ones; within each group, candidates are ranked by raw joint probability (the model's probability for the exact reconstruction). The length-normalised confidence score is reported separately so long fragments are not artificially preferred.

Each candidate contains the predicted fragment, full reconstructed SMILES, raw joint probability, confidence score, RDKit validity, canonical SMILES, and predictive entropy in nats/bits.

The parser accepts `[MASK]`, `<MASK>`, `*`, and `[*]` as the mask placeholder. A single placeholder searches every fragment length from 1 to `max_mask_span`; repeating the placeholder (for example `[MASK][MASK]`) fixes the fragment length explicitly.

Note: the prompt's literal example `CC(=O)Oc1ccc(*)c1` describes a five-membered aromatic ring and is not RDKit-valid; the tool still accepts it and correctly reports `rdkit_valid=False`. The corrected six-membered variant `CC(=O)Oc1ccc([MASK])cc1` is used below to show valid reconstructions.


In [ ]:
import json

from predict import get_bilstm_predictions, print_candidates

EXAMPLES = [
    "c1cc([MASK])ccc1C(=O)O",
    "CC(=O)Oc1ccc([MASK])cc1",
]

for masked_smiles in EXAMPLES:
    print("=" * 78)
    print("Masked SMILES:", masked_smiles)
    candidates = get_bilstm_predictions(masked_smiles, top_k=3)
    print_candidates(candidates)
    print(json.dumps(candidates, indent=2, ensure_ascii=False))


## 12. Interactive CLI

`predict.py` is a normal command-line tool. It reads `vocab.json` and the best checkpoint, then prints the required fields for each candidate. In a notebook, `input()` works, but the loop is disabled by default so **Run All** does not block.


In [ ]:
RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    from predict import interactive_cli

    interactive_cli(top_k=3)
else:
    print("Interactive prompt skipped (set RUN_INTERACTIVE = True to enable).")
    print("Equivalent shell command:")
    print('!python predict.py --masked-smiles "c1cc([MASK])ccc1C(=O)O" --top-k 3')


In [ ]:
!python predict.py --masked-smiles "c1cc([MASK])ccc1C(=O)O" --top-k 3


In [ ]:
!python predict.py --masked-smiles "CC(=O)Oc1ccc([MASK])cc1" --top-k 3 --json


## 13. OpenRouter integration payload

`get_bilstm_predictions(masked_smiles, top_k=3)` is the integration point for an OpenRouter orchestrator agent. The returned list is pure JSON-friendly Python (`str`, `float`, `bool`, `None`, `int`), so it can be embedded directly in a tool response.


In [ ]:
import json
from pathlib import Path

from predict import get_bilstm_predictions

masked_smiles = "c1cc([MASK])ccc1C(=O)O"
candidates = get_bilstm_predictions(masked_smiles, top_k=3)

orchestrator_payload = {
    "tool": "seqligase_bilstm_infill",
    "masked_smiles": masked_smiles,
    "candidates": candidates,
}

Path("artifacts/openrouter_predictions.json").write_text(
    json.dumps(orchestrator_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps(orchestrator_payload, indent=2, ensure_ascii=False))


## 14. Export the checkpoint

`export_model.py` packages the weights, vocabulary, model configuration, training configuration, and test metrics into a portable `.pt` bundle and a `.zip` for Kaggle download. The zip also includes a model card.


In [ ]:
from export_model import export_model

export_paths = export_model()
for name, path in export_paths.items():
    size_mb = path.stat().st_size / 1e6
    print(f"{name:12s} -> {path}  ({size_mb:.2f} MB)")

print("\nAll generated artefacts:")
for path in sorted(Path("artifacts").rglob("*")):
    if path.is_file():
        print(f"  {path}  ({path.stat().st_size / 1e6:.2f} MB)")


## 15. Troubleshooting and design notes

| Symptom | Fix |
|---|---|
| `Could not obtain the dataset` | Use **Add Data** on Kaggle and attach `ssshaun/clean-qm9-dataset-fro-seqligase-project`, or enable Internet so `kagglehub` can download it. |
| CUDA out of memory | Reduce `batch_size` to 128 (or 64) and keep `max_len=128`; the model is small enough for a T4. |
| Training is slow | Set `num_workers=2` (or 4) and confirm the runtime is GPU. Mixed precision is already enabled. |
| `Checkpoint not found` | Run the training cell before the inference/export cells. |
| RDKit import error | Re-run the environment cell; on Kaggle `rdkit` is normally pre-installed. |
| Want a quick run | Use `TrainConfig(epochs=2, max_samples=5000)`; the pipeline still exercises every stage. |

**Metric definitions**

- **Token accuracy (top-1):** fraction of masked positions where the argmax token matches the original.
- **Top-3 token accuracy:** fraction where the correct token is among the three highest-probability tokens.
- **Character-level accuracy:** mean `difflib.SequenceMatcher` similarity between the reconstructed and original SMILES.
- **Token precision (macro):** macro-averaged precision across token classes over masked positions.
- **Structural reconstruction pass rate:** fraction of reconstructions that are RDKit-valid **and** whose canonical SMILES equals the original's canonical SMILES.
- **RDKit validity rate:** fraction of reconstructions that RDKit can parse, regardless of whether they match the original.

**Reproducibility:** `seed=42` seeds Python, NumPy, and PyTorch; the provided Kaggle split is leakage-checked; validation/test masks are fixed; and the vocabulary is built from the training split only.
